# FiscalNinja: Complete Implementation Guide
## Step-by-Step Build Plan from Scratch

**Product:** OCR Receipt Parser for Small Trucking Companies  
**Target Market:** 1-25 truck fleets in North America  
**Revenue Goal:** $10k MRR (140 customers × $70 avg)  
**Timeline:** 30 days to MVP, 12-24 months to $10k MRR  
**Startup Cost:** <$200

---

## 📋 Implementation Overview

This notebook provides a complete step-by-step guide to build FiscalNinja from scratch. Each phase includes:
- **Detailed tasks** with specific deliverables
- **LLM prompts** you can use to generate code, designs, and content
- **Technical specifications** for every component
- **Testing criteria** to validate each step
- **Time estimates** to track progress

---

## 🎯 Product Vision Reminder

**What FiscalNinja Does:**
Truck drivers snap photos of fuel receipts → App uses OCR to extract date, amount, vendor → Owner downloads Excel file with all expenses organized by driver/truck → Saves 20+ hours/month of manual data entry.

**Core Value Proposition:**
Transform receipt chaos into organized expense reports in 5 minutes/week instead of 20 hours/month.

---

## 📊 Tech Stack Summary

| Layer | Technology | Why This Choice |
|-------|-----------|-----------------|
| **Frontend** | Next.js + React + Tailwind CSS | Modern, fast, mobile-responsive PWA |
| **Backend** | Next.js API Routes | Serverless, scales automatically, no separate backend needed |
| **Database** | Supabase (PostgreSQL) | Free tier, built-in auth, real-time, file storage |
| **OCR Engine** | Google Cloud Vision API | $0.01/image, 85-95% accuracy on receipts, 1k free/month |
| **File Storage** | Supabase Storage (S3-compatible) | Integrated with database, $0 for free tier |
| **Authentication** | Supabase Auth | Built-in, supports email/password + social logins |
| **Payments** | Stripe | Industry standard, easy subscription billing |
| **Hosting** | Vercel | Free tier, automatic deployments, global CDN |
| **Analytics** | Vercel Analytics + Mixpanel | Free tiers, track user behavior |

---

## 📅 30-Day Build Timeline

### Week 1: Foundation & Setup
- **Days 1-2:** Project setup, database schema, authentication
- **Days 3-4:** Receipt upload UI, file storage
- **Days 5-7:** OCR integration, data extraction

### Week 2: Core Features
- **Days 8-10:** Receipt management dashboard
- **Days 11-12:** Excel export functionality
- **Days 13-14:** Basic reporting & filtering

### Week 3: Business Logic
- **Days 15-17:** Multi-user support (drivers, owners, managers)
- **Days 18-19:** Subscription billing (Stripe integration)
- **Days 20-21:** Email notifications & alerts

### Week 4: Polish & Launch
- **Days 22-24:** Mobile optimization (PWA features)
- **Days 25-26:** Testing, bug fixes, security audit
- **Days 27-28:** Landing page, marketing materials
- **Days 29-30:** Beta testing with 3-5 pilot customers

---

## 🚀 Let's Begin Implementation

The following sections break down each phase into actionable steps with LLM prompts you can copy-paste to accelerate development.

# PHASE 1: Project Setup & Infrastructure (Days 1-2)

## Step 1.1: Create Next.js Project

**Objective:** Initialize a new Next.js application with TypeScript and Tailwind CSS.

**Time Estimate:** 30 minutes

### Tasks:
1. Create new Next.js project
2. Install dependencies (Tailwind, UI library, utilities)
3. Configure TypeScript
4. Set up folder structure

### LLM Prompt for Code Generation:

```
Create a Next.js 14 project setup script with the following requirements:

1. Use TypeScript
2. Include Tailwind CSS with default configuration
3. Install these dependencies:
   - shadcn/ui components (button, card, input, table, dialog)
   - react-dropzone (for file uploads)
   - date-fns (for date formatting)
   - lucide-react (for icons)
   - zustand (for state management)

4. Create this folder structure:
   /src
     /app
       /api
       /dashboard
       /auth
     /components
       /ui
       /receipts
       /layout
     /lib
       /utils
       /db
       /ocr
     /types
     /hooks

5. Include a basic layout component with navigation header

Provide: 
- Complete package.json
- All necessary configuration files (tsconfig.json, tailwind.config.ts, next.config.js)
- Initial folder structure with placeholder files
```

### Manual Commands to Run:

```bash
# Navigate to FiscalNinja folder
cd c:\Users\trica\OneDrive\Área de Trabalho\Projects\FiscalNinja

# Create Next.js app
npx create-next-app@latest . --typescript --tailwind --app --use-npm

# Install additional dependencies
npm install @supabase/supabase-js @supabase/auth-helpers-nextjs
npm install react-dropzone date-fns lucide-react zustand
npm install xlsx
npm install stripe @stripe/stripe-js

# Install shadcn/ui
npx shadcn-ui@latest init

# Add shadcn components
npx shadcn-ui@latest add button card input table dialog select label textarea badge alert
```

### Validation Checklist:
- [ ] Next.js dev server runs without errors (`npm run dev`)
- [ ] Tailwind CSS is working (test with a colored div)
- [ ] TypeScript compilation works
- [ ] All dependencies installed successfully

---

## Step 1.2: Set Up Supabase Project

**Objective:** Create Supabase project for database, authentication, and file storage.

**Time Estimate:** 45 minutes

### Tasks:
1. Create Supabase account & project
2. Set up database schema
3. Configure authentication
4. Set up storage bucket for receipts

### LLM Prompt for Database Schema:

```
Create a PostgreSQL database schema for a receipt management SaaS for trucking companies with these requirements:

TABLES NEEDED:

1. profiles (extends Supabase auth.users)
   - id (UUID, references auth.users)
   - company_name (text)
   - email (text)
   - subscription_tier (enum: 'solo', 'fleet', 'enterprise')
   - subscription_status (enum: 'active', 'cancelled', 'past_due')
   - stripe_customer_id (text, nullable)
   - created_at (timestamp)
   - updated_at (timestamp)

2. trucks
   - id (UUID, primary key)
   - user_id (UUID, references profiles.id)
   - truck_number (text, e.g., "Truck 101")
   - license_plate (text, nullable)
   - vin (text, nullable)
   - active (boolean, default true)
   - created_at (timestamp)

3. drivers
   - id (UUID, primary key)
   - user_id (UUID, references profiles.id)
   - name (text)
   - email (text, nullable)
   - phone (text, nullable)
   - active (boolean, default true)
   - created_at (timestamp)

4. receipts
   - id (UUID, primary key)
   - user_id (UUID, references profiles.id)
   - driver_id (UUID, references drivers.id, nullable)
   - truck_id (UUID, references trucks.id, nullable)
   - image_url (text, Supabase storage path)
   - receipt_date (date)
   - vendor (text)
   - amount (decimal)
   - tax_amount (decimal, nullable)
   - category (enum: 'fuel', 'tolls', 'maintenance', 'insurance', 'other')
   - payment_method (enum: 'cash', 'credit', 'debit', 'company_card', 'other')
   - notes (text, nullable)
   - ocr_confidence (decimal, 0-100)
   - needs_review (boolean, default false)
   - reviewed_at (timestamp, nullable)
   - created_at (timestamp)
   - updated_at (timestamp)

5. expense_categories
   - id (UUID, primary key)
   - user_id (UUID, references profiles.id)
   - name (text)
   - tax_deductible (boolean, default true)
   - created_at (timestamp)

REQUIREMENTS:
- Include Row Level Security (RLS) policies for multi-tenant isolation
- Add indexes on frequently queried columns (user_id, receipt_date, truck_id, driver_id)
- Include database functions for common queries (total expenses by month, expenses by driver)
- Add triggers for updated_at timestamps

Provide:
1. Complete SQL schema with CREATE TABLE statements
2. RLS policies for each table
3. Useful database functions
4. Index creation statements
```

### Manual Steps in Supabase Dashboard:

1. **Create Project:**
   - Go to https://supabase.com
   - Click "New Project"
   - Name: "FiscalNinja"
   - Database Password: (save this securely!)
   - Region: Choose closest to your target market

2. **Run SQL Schema:**
   - Navigate to SQL Editor in Supabase dashboard
   - Copy the generated SQL from LLM
   - Execute the schema creation

3. **Configure Storage:**
   - Go to Storage section
   - Create new bucket: `receipts`
   - Make it private (requires authentication)
   - Set file size limit: 10MB per file
   - Allowed file types: image/jpeg, image/png, image/webp, application/pdf

4. **Get API Keys:**
   - Go to Project Settings → API
   - Copy: `Project URL` and `anon public` key
   - Save these for environment variables

### LLM Prompt for Environment Variables:

```
Create a .env.local file template for a Next.js app using Supabase with these requirements:

1. Supabase connection (URL, anon key, service role key)
2. Google Cloud Vision API credentials
3. Stripe keys (publishable and secret)
4. App URL for redirects
5. Include comments explaining each variable

Also create a .env.example file (without real values) for version control.
```

### Validation Checklist:
- [ ] Supabase project created
- [ ] Database schema executed without errors
- [ ] Storage bucket `receipts` created
- [ ] Environment variables added to `.env.local`
- [ ] Connection test successful (use Supabase client in test script)

---

## Step 1.3: Set Up Authentication

**Objective:** Implement user sign-up, login, and session management.

**Time Estimate:** 2 hours

### Tasks:
1. Create authentication pages (sign up, login, forgot password)
2. Implement Supabase Auth helpers
3. Add protected route middleware
4. Create user profile setup flow

### LLM Prompt for Auth Pages:

```
Create a complete authentication system for a Next.js 14 app using Supabase Auth with these pages:

1. /auth/login
   - Email/password form
   - "Forgot password" link
   - "Sign up" link
   - Error handling for invalid credentials
   - Redirect to /dashboard on success
   - Use shadcn/ui components (Card, Input, Button)

2. /auth/signup
   - Email, password, company name fields
   - Password strength indicator
   - Terms of service checkbox
   - Automatic profile creation in profiles table after signup
   - Email verification required
   - Redirect to /auth/verify-email

3. /auth/forgot-password
   - Email input
   - Send password reset email via Supabase Auth
   - Success message with next steps

4. /auth/reset-password (accessed via email link)
   - New password form
   - Confirm password validation
   - Update password via Supabase

5. /auth/verify-email
   - Message: "Check your email to verify account"
   - Resend verification email button

REQUIREMENTS:
- Use Supabase auth helpers (@supabase/auth-helpers-nextjs)
- Implement proper error handling with user-friendly messages
- Add loading states for all forms
- Use TypeScript with proper types
- Follow shadcn/ui design patterns
- Include form validation (email format, password strength)
- Add CSRF protection

Also create:
- Middleware for protecting dashboard routes (redirect to /auth/login if not authenticated)
- useUser custom hook for accessing current user
- Logout functionality

Provide complete code for all files.
```

### Additional LLM Prompt for Profile Setup:

```
Create a first-time setup flow at /onboarding for new users with these steps:

Step 1: Company Information
- Company name (pre-filled from signup)
- Phone number
- Address (optional)

Step 2: Add First Truck
- Truck number (e.g., "Truck 101")
- License plate (optional)
- Button to add multiple trucks

Step 3: Add First Driver
- Driver name
- Email (optional)
- Phone (optional)
- Button to add multiple drivers

Step 4: Subscription Selection
- Show 3 tiers: Solo ($29), Fleet ($79), Enterprise ($149)
- Feature comparison table
- "Start 14-day free trial" button

REQUIREMENTS:
- Multi-step form with progress indicator
- Data saved to database at each step
- Skip option for steps 2-3 (can add later)
- Redirect to /dashboard after completion
- Use React Hook Form for form management
- Proper TypeScript types

Provide complete code.
```

### Validation Checklist:
- [ ] Users can sign up with email/password
- [ ] Email verification required before login
- [ ] Login redirects to dashboard
- [ ] Logout works correctly
- [ ] Forgot password flow works
- [ ] Protected routes redirect unauthenticated users
- [ ] Onboarding flow saves data to database
- [ ] User profile created in `profiles` table

---

## Step 1.4: Configure Google Cloud Vision API

**Objective:** Set up OCR service for receipt processing.

**Time Estimate:** 1 hour

### Tasks:
1. Create Google Cloud project
2. Enable Vision API
3. Create service account & download credentials
4. Test OCR with sample receipt

### Manual Steps:

1. **Create Google Cloud Project:**
   - Go to https://console.cloud.google.com
   - Create new project: "FiscalNinja"
   - Enable billing (required for API access)

2. **Enable Vision API:**
   - Navigate to APIs & Services → Library
   - Search "Cloud Vision API"
   - Click Enable

3. **Create Service Account:**
   - Go to IAM & Admin → Service Accounts
   - Create service account: "fiscalninja-ocr"
   - Grant role: "Cloud Vision API User"
   - Create key (JSON format)
   - Download JSON file → save as `google-credentials.json` in project root
   - Add to `.gitignore`

4. **Add to Environment Variables:**
   ```
   GOOGLE_APPLICATION_CREDENTIALS=./google-credentials.json
   ```

### LLM Prompt for OCR Integration:

```
Create a TypeScript module for receipt OCR processing using Google Cloud Vision API with these requirements:

FILE: /src/lib/ocr/vision.ts

FUNCTIONS NEEDED:

1. processReceipt(imageBuffer: Buffer): Promise<ReceiptData>
   - Send image to Google Vision API (Text Detection)
   - Parse response to extract:
     - Total amount (look for keywords: "total", "amount", "$")
     - Date (various formats: MM/DD/YYYY, DD-MM-YYYY, etc.)
     - Vendor/merchant name (usually at top of receipt)
     - Tax amount (if present)
   - Return structured data with confidence scores
   - Handle errors gracefully

2. validateReceiptImage(file: File): Promise<boolean>
   - Check file size (<10MB)
   - Check file type (JPEG, PNG, WebP, PDF)
   - Check image dimensions (min 200x200px)
   - Return validation result with error messages

3. enhanceImageQuality(imageBuffer: Buffer): Promise<Buffer>
   - Auto-rotate if needed
   - Increase contrast
   - Sharpen edges
   - Convert to grayscale
   - Use sharp library for image processing

TYPES:
```typescript
interface ReceiptData {
  vendor: string;
  total: number;
  date: Date;
  taxAmount?: number;
  rawText: string;
  confidence: number; // 0-100
  needsReview: boolean; // true if confidence < 80%
}
```

ERROR HANDLING:
- Handle Vision API quota limits
- Retry logic for transient failures
- Fallback to manual entry if OCR fails completely

TESTING:
- Include test function with sample receipt image
- Log extracted data to console

Provide complete implementation with proper TypeScript types and error handling.
```

### Test OCR Setup:

```
Create a test script at /src/lib/ocr/test-ocr.ts that:

1. Reads a sample receipt image from /public/test-receipt.jpg
2. Calls processReceipt()
3. Logs extracted data to console
4. Validates that total, date, and vendor were extracted
5. Shows confidence score

Provide complete code and instructions to run the test.
```

### Validation Checklist:
- [ ] Google Cloud project created
- [ ] Vision API enabled
- [ ] Service account credentials downloaded
- [ ] OCR test script runs successfully
- [ ] Can extract total, date, vendor from test receipt
- [ ] Confidence score is calculated
- [ ] Error handling works (test with invalid image)

---

## Summary: Phase 1 Complete ✅

**What You've Built:**
- ✅ Next.js project with TypeScript & Tailwind CSS
- ✅ Supabase database with complete schema
- ✅ Authentication system (signup, login, password reset)
- ✅ User onboarding flow
- ✅ Google Cloud Vision API integration
- ✅ OCR testing confirmed working

**Ready for Phase 2:** Receipt Upload & Management

---

## 🎯 Quick Start Commands (Copy-Paste)

```bash
# Clone template or start fresh
cd c:\Users\trica\OneDrive\Área de Trabalho\Projects\FiscalNinja

# Install dependencies
npm install

# Set up environment variables (fill in values)
cp .env.example .env.local

# Run development server
npm run dev

# Test OCR
npm run test:ocr

# Open browser
start http://localhost:3000
```

# PHASE 2: Receipt Upload & OCR Processing (Days 3-7)

## Step 2.1: Build Receipt Upload UI

**Objective:** Create mobile-friendly interface for uploading receipt photos.

**Time Estimate:** 3 hours

### LLM Prompt for Upload Component:

```
Create a React component for receipt upload with these requirements:

FILE: /src/components/receipts/ReceiptUpload.tsx

FEATURES:

1. Drag-and-drop zone for desktop
2. Camera capture for mobile devices
3. Multi-file upload support (up to 10 receipts at once)
4. Image preview before upload
5. Progress indicator during upload
6. Success/error messages
7. Optional metadata input:
   - Select driver (dropdown from drivers table)
   - Select truck (dropdown from trucks table)
   - Category (dropdown: fuel, tolls, maintenance, etc.)
   - Notes (textarea, optional)

UI REQUIREMENTS:
- Use react-dropzone for file handling
- Use shadcn/ui components (Card, Button, Select, Textarea)
- Mobile-responsive design (works on phone cameras)
- Show file size and type validation errors
- Loading state with spinner during upload
- Thumbnail preview of uploaded images

WORKFLOW:
1. User selects/drops image
2. Validate file (size <10MB, type image/*)
3. Show preview with metadata form
4. On submit:
   a. Upload image to Supabase Storage (receipts bucket)
   b. Get public URL
   c. Send to OCR processing API
   d. Show "Processing..." state
   e. On OCR complete, show extracted data for review
   f. Save to database

ERROR HANDLING:
- File too large
- Invalid file type
- Upload failed
- OCR failed (allow manual entry)

MOBILE OPTIMIZATION:
- Large touch targets (buttons min 44px height)
- Camera input: <input type="file" accept="image/*" capture="environment">
- Auto-rotate images if needed

Provide complete component code with TypeScript types.
```

### LLM Prompt for File Upload Service:

```
Create a file upload service for Supabase Storage with these functions:

FILE: /src/lib/storage/upload.ts

FUNCTIONS:

1. uploadReceipt(file: File, userId: string): Promise<string>
   - Generate unique filename: {userId}/{timestamp}_{random}.{ext}
   - Upload to Supabase Storage bucket "receipts"
   - Return public URL
   - Handle errors (quota exceeded, network failure)

2. deleteReceipt(filePath: string): Promise<void>
   - Delete file from Supabase Storage
   - Used when user deletes receipt

3. getSignedUrl(filePath: string, expiresIn: number): Promise<string>
   - Generate temporary signed URL for private files
   - Used for displaying images in dashboard

4. compressImage(file: File, maxSizeMB: number): Promise<File>
   - Compress large images before upload
   - Use browser-image-compression library
   - Maintain quality while reducing file size

Include proper error handling and TypeScript types.
```

### Validation Checklist:
- [ ] Upload component renders on /dashboard/upload page
- [ ] Drag-and-drop works on desktop
- [ ] Camera capture works on mobile
- [ ] File validation works (reject >10MB, non-images)
- [ ] Images upload to Supabase Storage
- [ ] Preview shows uploaded images
- [ ] Metadata form allows selecting driver/truck

---

## Step 2.2: Build OCR Processing API

**Objective:** Create API endpoint to process receipts with OCR.

**Time Estimate:** 4 hours

### LLM Prompt for OCR API Route:

```
Create a Next.js API route for OCR processing with these requirements:

FILE: /src/app/api/receipts/process/route.ts

ENDPOINT: POST /api/receipts/process

REQUEST BODY:
```typescript
{
  imageUrl: string;  // Supabase Storage URL
  userId: string;
  driverId?: string;
  truckId?: string;
  category?: string;
}
```

PROCESSING FLOW:
1. Validate request (check auth, user owns the receipt)
2. Download image from Supabase Storage
3. Enhance image quality (increase contrast, rotate if needed)
4. Send to Google Cloud Vision API for text detection
5. Parse OCR response to extract:
   - Total amount (regex patterns for $XX.XX, XX.XX, etc.)
   - Date (handle multiple formats: MM/DD/YYYY, MM-DD-YY, etc.)
   - Vendor name (usually first few lines of text)
   - Tax amount (if present)
6. Calculate confidence score (0-100)
7. Flag for review if confidence < 80%
8. Save to receipts table in database
9. Return structured data

RESPONSE:
```typescript
{
  success: boolean;
  receipt: {
    id: string;
    vendor: string;
    amount: number;
    date: string;
    taxAmount?: number;
    category: string;
    confidence: number;
    needsReview: boolean;
    imageUrl: string;
  };
  error?: string;
}
```

ERROR HANDLING:
- Image download failed
- OCR API quota exceeded
- Cannot extract required fields (amount/date)
- Database save failed

SMART PARSING:
- Use regex patterns for common receipt formats
- Handle multiple date formats
- Detect currency symbols ($, USD, etc.)
- Clean up vendor names (remove extra whitespace, special chars)
- Validate extracted data (amount > 0, date not in future)

FALLBACK:
- If OCR confidence < 50%, flag for manual entry
- Store raw OCR text for debugging

Include rate limiting (max 50 requests/minute per user).

Provide complete API route code with TypeScript types.
```

### LLM Prompt for OCR Parser Module:

```
Create a specialized receipt parser module with these functions:

FILE: /src/lib/ocr/parser.ts

FUNCTIONS:

1. extractAmount(text: string): { amount: number; confidence: number }
   - Search for patterns: "$XX.XX", "Total: XX.XX", "TOTAL XX.XX"
   - Handle variations: spaces, commas in numbers ($1,234.56)
   - Return highest confidence match
   - Confidence based on context (words "total", "amount" nearby)

2. extractDate(text: string): { date: Date | null; confidence: number }
   - Patterns: MM/DD/YYYY, DD-MM-YYYY, YYYY-MM-DD, Mon DD YYYY
   - Validate date is reasonable (not in future, not >1 year old)
   - Return most likely date

3. extractVendor(text: string): { vendor: string; confidence: number }
   - Usually first 1-3 lines of receipt
   - Common vendors: Shell, Chevron, Love's, Pilot Flying J
   - Clean up (remove addresses, phone numbers)
   - Return cleaned vendor name

4. extractTax(text: string): { tax: number | null; confidence: number }
   - Search for: "Tax:", "GST:", "Sales Tax"
   - Return tax amount if found

5. categorizeReceipt(vendor: string, text: string): string
   - Auto-detect category based on vendor and keywords
   - Categories: fuel, tolls, maintenance, insurance, other
   - Keywords: "gas", "diesel" → fuel; "toll" → tolls; "oil change", "tire" → maintenance

TESTING:
- Include unit tests with sample receipt text
- Test edge cases (blurry OCR, missing data)

Provide complete module with TypeScript types and comprehensive regex patterns.
```

### Test Cases for OCR:

```
Create test cases for OCR parsing at /src/lib/ocr/__tests__/parser.test.ts

Test these scenarios:

1. Standard fuel receipt (Shell, Chevron)
   - Clear amount: $125.50
   - Date: 02/10/2026
   - Vendor: Shell Gas Station

2. Toll receipt
   - Amount: $15.75
   - E-ZPass transaction

3. Maintenance receipt
   - Oil change: $89.99
   - Jiffy Lube

4. Handwritten receipt (low quality OCR)
   - Blurry text
   - Should flag for manual review

5. Missing data
   - Amount visible, date missing
   - Should extract what's available

Provide Jest test suite with assertions.
```

### Validation Checklist:
- [ ] API route `/api/receipts/process` works
- [ ] Can process uploaded image and extract data
- [ ] Amount extraction works (test with 10+ receipts)
- [ ] Date extraction works (multiple formats)
- [ ] Vendor extraction works
- [ ] Confidence score calculated correctly
- [ ] Low-confidence receipts flagged for review
- [ ] Data saved to database
- [ ] Error handling works (invalid image, OCR failure)

---

## Step 2.3: Build Receipt Review Interface

**Objective:** Allow users to review and correct OCR results before finalizing.

**Time Estimate:** 3 hours

### LLM Prompt for Review Component:

```
Create a receipt review component for verifying OCR results:

FILE: /src/components/receipts/ReceiptReview.tsx

PROPS:
- receipt: Receipt object from database
- onSave: (updatedReceipt: Receipt) => void
- onCancel: () => void

FEATURES:

1. Side-by-side layout (desktop) or stacked (mobile):
   - LEFT: Receipt image (zoomable, rotatable)
   - RIGHT: Editable form with OCR data

2. Editable fields:
   - Vendor (text input, pre-filled from OCR)
   - Amount (number input, pre-filled)
   - Date (date picker, pre-filled)
   - Tax Amount (number input, optional)
   - Category (dropdown)
   - Driver (dropdown, can change)
   - Truck (dropdown, can change)
   - Payment Method (dropdown)
   - Notes (textarea)

3. Confidence indicators:
   - Green badge: ≥80% confidence (High)
   - Yellow badge: 50-79% confidence (Medium - Review Recommended)
   - Red badge: <50% confidence (Low - Manual Entry Required)

4. Quick actions:
   - "Save" button (saves corrections to database)
   - "Delete" button (delete receipt + image)
   - "Skip" button (save without changes)

5. Keyboard shortcuts:
   - Enter: Save
   - Esc: Cancel
   - Tab: Navigate fields

UI REQUIREMENTS:
- Image viewer with zoom controls
- Form validation (amount > 0, date required)
- Show which fields were auto-extracted vs manually entered
- Highlight fields that need review (low confidence)
- Auto-save draft every 30 seconds
- Responsive design

Use shadcn/ui components (Input, Select, Button, Card, Badge).

Provide complete component code with TypeScript.
```

### LLM Prompt for Bulk Review:

```
Create a bulk review interface for processing multiple receipts:

FILE: /src/app/dashboard/receipts/review/page.tsx

FEATURES:

1. Show list of receipts needing review (confidence < 80%)
2. Display count: "5 receipts need review"
3. Card grid layout with:
   - Thumbnail image
   - Extracted data preview
   - Confidence badge
   - "Review" button

4. Click "Review" → opens ReceiptReview modal/page
5. After review, move to next receipt automatically
6. Progress indicator: "2 of 5 reviewed"

7. Filters:
   - Show all / Only needs review
   - Date range
   - Driver filter
   - Category filter

8. Bulk actions:
   - "Approve all" (for high-confidence batches)
   - "Delete selected"
   - "Export to Excel"

Use React Server Components for data fetching.

Provide complete page code.
```

### Validation Checklist:
- [ ] Review interface displays receipt image + OCR data
- [ ] Can edit all fields (vendor, amount, date, etc.)
- [ ] Confidence badges show correct colors
- [ ] Save updates database
- [ ] Delete removes receipt + image from storage
- [ ] Bulk review shows list of flagged receipts
- [ ] Can navigate through multiple receipts
- [ ] Auto-save works

---

## Step 2.4: Implement Receipt Dashboard

**Objective:** Create main dashboard for viewing all receipts.

**Time Estimate:** 4 hours

### LLM Prompt for Dashboard:

```
Create a comprehensive receipt dashboard:

FILE: /src/app/dashboard/page.tsx

SECTIONS:

1. OVERVIEW STATS (Top of page)
   - Total receipts this month
   - Total expenses this month (sum of amounts)
   - Receipts needing review (confidence < 80%)
   - Average expense per receipt

2. QUICK FILTERS
   - Date range picker (This week, This month, Last 30 days, Custom)
   - Driver filter (multiselect dropdown)
   - Truck filter (multiselect dropdown)
   - Category filter (multiselect)
   - Search bar (search vendor, amount, notes)

3. RECEIPTS TABLE
   Columns:
   - Date (sortable)
   - Thumbnail (clickable to enlarge)
   - Vendor
   - Amount (formatted as currency)
   - Category (with colored badge)
   - Driver
   - Truck
   - Confidence (badge)
   - Actions (View, Edit, Delete)

   Features:
   - Sortable columns
   - Pagination (20 per page)
   - Row selection (checkboxes)
   - Bulk actions (Delete selected, Export selected)

4. CHART (Optional - can add later)
   - Bar chart: Expenses by category this month
   - Use recharts library

DATA FETCHING:
- Use React Server Components
- Query Supabase with filters applied
- Include related data (driver name, truck number)
- Optimize query performance (indexes)

UI:
- Use shadcn/ui Table component
- Responsive (card view on mobile)
- Loading states (skeletons)
- Empty state: "No receipts yet - upload your first receipt"

Provide complete dashboard code with TypeScript.
```

### LLM Prompt for Receipt Detail Modal:

```
Create a modal for viewing full receipt details:

FILE: /src/components/receipts/ReceiptDetail.tsx

FEATURES:

1. Large image viewer
   - Full-size receipt image
   - Zoom controls (+/-)
   - Rotate controls
   - Download original button

2. Receipt information:
   - All fields (vendor, amount, date, category, driver, truck, etc.)
   - Read-only view
   - "Edit" button → opens ReceiptReview component
   - OCR metadata (confidence, extracted text, processing timestamp)

3. Actions:
   - Edit
   - Delete (with confirmation)
   - Download image
   - Add to export queue

Use shadcn/ui Dialog component.

Provide complete modal code.
```

### Validation Checklist:
- [ ] Dashboard displays all receipts in table format
- [ ] Stats cards show correct totals
- [ ] Filters work (date range, driver, truck, category)
- [ ] Search finds receipts by vendor/notes
- [ ] Table sorting works
- [ ] Pagination works
- [ ] Detail modal shows full receipt info
- [ ] Can edit from modal
- [ ] Can delete receipts
- [ ] Mobile responsive (card view)

---

## Summary: Phase 2 Complete ✅

**What You've Built:**
- ✅ Receipt upload interface (desktop + mobile)
- ✅ OCR processing API with Google Vision
- ✅ Smart receipt parser (amount, date, vendor extraction)
- ✅ Review interface for correcting OCR errors
- ✅ Comprehensive dashboard with filters
- ✅ Receipt detail view

**Ready for Phase 3:** Excel Export & Reporting

---

## 🧪 Testing Phase 2

### Test Checklist:

**Upload Flow:**
1. [ ] Upload receipt via drag-and-drop
2. [ ] Upload receipt via camera on phone
3. [ ] Upload multiple receipts at once
4. [ ] Validate file size limit (reject >10MB)
5. [ ] Validate file type (reject non-images)

**OCR Processing:**
1. [ ] OCR extracts amount correctly (test 10 receipts)
2. [ ] OCR extracts date correctly (test various formats)
3. [ ] OCR extracts vendor correctly
4. [ ] Low-confidence receipts flagged for review
5. [ ] Manual override works for failed OCR

**Dashboard:**
1. [ ] All receipts displayed in table
2. [ ] Filters work correctly
3. [ ] Search finds receipts
4. [ ] Sorting works
5. [ ] Stats cards show correct totals
6. [ ] Mobile view works properly

**Performance:**
1. [ ] OCR processing completes in <10 seconds
2. [ ] Dashboard loads in <2 seconds
3. [ ] Image uploads in <5 seconds
4. [ ] No memory leaks (test uploading 50 receipts)

# PHASE 3: Excel Export & Reporting (Days 8-14)

## Step 3.1: Build Excel Export Functionality

**Objective:** Generate downloadable Excel files with expense reports.

**Time Estimate:** 3 hours

### LLM Prompt for Excel Export:

```
Create an Excel export service with these requirements:

FILE: /src/lib/export/excel.ts

FUNCTIONS:

1. generateExpenseReport(receipts: Receipt[], options: ExportOptions): Promise<Buffer>
   
   OPTIONS:
   ```typescript
   interface ExportOptions {
     dateRange: { start: Date; end: Date };
     groupBy?: 'driver' | 'truck' | 'category' | 'none';
     includeSummary: boolean;
     includeImages: boolean; // Add image thumbnails
     format: 'xlsx' | 'csv';
   }
   ```

   EXCEL STRUCTURE:
   
   Sheet 1: "Expense Report"
   Columns:
   - Date (formatted MM/DD/YYYY)
   - Vendor
   - Category
   - Amount (formatted as currency $XX.XX)
   - Tax (if available)
   - Payment Method
   - Driver Name
   - Truck Number
   - Notes
   - Receipt Image (hyperlink to Supabase URL)
   
   Features:
   - Header row with bold formatting
   - Alternating row colors (light gray/white)
   - Total row at bottom (sum of amounts)
   - Auto-fit column widths
   - Freeze header row
   - Currency formatting for amount columns
   
   Sheet 2: "Summary" (if includeSummary: true)
   - Total expenses by category (pie chart data)
   - Total expenses by driver (table)
   - Total expenses by truck (table)
   - Grand total
   - Date range
   
   Sheet 3: "Tax Summary" (optional)
   - Tax-deductible expenses by category
   - Total tax paid
   - Format for IRS Schedule C

2. generateCSV(receipts: Receipt[]): string
   - Simple CSV export
   - Same columns as Excel
   - Quoted fields for special characters
   - UTF-8 encoding

3. generateQuickBooksImport(receipts: Receipt[]): Buffer
   - Format compatible with QuickBooks CSV import
   - Columns: Date, Description, Account, Amount, Memo
   - Map categories to QuickBooks expense accounts

Use xlsx library (npm install xlsx).

ERROR HANDLING:
- Handle large datasets (>1000 receipts) efficiently
- Stream processing for memory efficiency
- Validate data before export

Provide complete implementation with TypeScript types.
```

### LLM Prompt for Export UI:

```
Create an export modal component:

FILE: /src/components/export/ExportModal.tsx

FEATURES:

1. Export configuration form:
   - Date range selector (preset options + custom range)
   - Group by dropdown (None, Driver, Truck, Category)
   - Checkboxes:
     □ Include summary sheet
     □ Include receipt images (as hyperlinks)
     □ Tax summary (IRS-ready)
   - Format selection (Excel .xlsx, CSV, QuickBooks CSV)
   
2. Preview section:
   - Show count: "Exporting 47 receipts"
   - Total amount: "$3,245.89"
   - Date range: "Feb 1 - Feb 28, 2026"

3. Export button:
   - Triggers download
   - Shows progress bar during generation
   - Success message: "Report downloaded successfully"

4. Quick export presets:
   - "This Month" button
   - "Last Month" button
   - "Tax Year 2026" button
   - "All Time" button

UI:
- Use shadcn/ui Dialog, Select, Checkbox, Button
- Responsive design
- Loading state during export generation

Provide complete modal code.
```

### LLM Prompt for Export API Route:

```
Create an API route for generating exports:

FILE: /src/app/api/exports/generate/route.ts

ENDPOINT: POST /api/exports/generate

REQUEST BODY:
```typescript
{
  userId: string;
  dateRange: { start: string; end: string };
  driverIds?: string[];
  truckIds?: string[];
  categoryIds?: string[];
  groupBy?: 'driver' | 'truck' | 'category';
  includeSummary: boolean;
  format: 'xlsx' | 'csv' | 'quickbooks';
}
```

PROCESSING:
1. Authenticate user
2. Fetch receipts from database with filters
3. Include related data (driver names, truck numbers)
4. Generate export file using export service
5. Return file as download

RESPONSE:
- Content-Type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet (for Excel)
- Content-Disposition: attachment; filename="fiscalninja-expenses-{date}.xlsx"
- Binary file stream

RATE LIMITING:
- Max 10 exports per hour per user
- Prevent abuse

OPTIMIZATION:
- Cache common exports (e.g., "This Month") for 1 hour
- Use streaming for large datasets
- Compress output

Provide complete API route code.
```

### Validation Checklist:
- [ ] Excel export generates valid .xlsx file
- [ ] CSV export works correctly
- [ ] QuickBooks format exports properly
- [ ] All receipt data included in export
- [ ] Summary sheet calculates totals correctly
- [ ] Grouping works (by driver, truck, category)
- [ ] Date range filtering works
- [ ] Large exports (500+ receipts) work without timeout
- [ ] Files download with correct filename
- [ ] Export modal UI works smoothly

---

## Step 3.2: Build QuickBooks Integration

**Objective:** Sync receipts directly to QuickBooks Online.

**Time Estimate:** 5 hours

### LLM Prompt for QuickBooks OAuth:

```
Create QuickBooks Online OAuth integration:

FILE: /src/lib/integrations/quickbooks/auth.ts

REQUIREMENTS:

1. QuickBooks OAuth 2.0 setup:
   - Create app at developer.intuit.com
   - Get Client ID and Client Secret
   - Redirect URI: https://fiscalninja.com/api/integrations/quickbooks/callback

2. Functions needed:

   a. getAuthUrl(userId: string): string
      - Generate QuickBooks OAuth URL
      - Include state parameter (userId encrypted)
      - Scopes: com.intuit.quickbooks.accounting

   b. handleCallback(code: string, state: string): Promise<TokenData>
      - Exchange code for access token + refresh token
      - Decrypt state to get userId
      - Save tokens to database (encrypted)
      - Return company info

   c. refreshAccessToken(userId: string): Promise<string>
      - Refresh expired access token
      - QuickBooks tokens expire after 60 minutes
      - Update database with new token

   d. disconnectQuickBooks(userId: string): Promise<void>
      - Revoke tokens
      - Remove from database

3. Database additions:
   - Table: quickbooks_connections
     - user_id (UUID, references profiles.id)
     - realm_id (QuickBooks company ID)
     - access_token (encrypted text)
     - refresh_token (encrypted text)
     - token_expires_at (timestamp)
     - connected_at (timestamp)

Provide complete OAuth implementation with security best practices.
```

### LLM Prompt for QuickBooks Sync:

```
Create QuickBooks sync service:

FILE: /src/lib/integrations/quickbooks/sync.ts

FUNCTIONS:

1. syncReceiptToQuickBooks(receiptId: string): Promise<void>
   
   WORKFLOW:
   - Fetch receipt from database
   - Get QuickBooks connection for user
   - Refresh token if needed
   - Create expense in QuickBooks:
     ```
     POST https://quickbooks.api.intuit.com/v3/company/{realmId}/expense
     {
       "PaymentType": "Cash",
       "AccountRef": { "value": "80" }, // Expense account
       "EntityRef": { "value": "{vendorId}" }, // Vendor
       "TxnDate": "2026-02-10",
       "Line": [{
         "Amount": 125.50,
         "DetailType": "AccountBasedExpenseLineDetail",
         "AccountBasedExpenseLineDetail": {
           "AccountRef": { "value": "85" }, // Fuel account
         }
       }]
     }
     ```
   - Save QuickBooks expense ID to receipts table
   - Handle errors (duplicate, invalid vendor)

2. batchSyncReceipts(receiptIds: string[]): Promise<SyncResult[]>
   - Sync multiple receipts
   - Use QuickBooks batch API (max 30 per request)
   - Retry failed syncs
   - Return status for each receipt

3. getQuickBooksAccounts(userId: string): Promise<Account[]>
   - Fetch chart of accounts from QuickBooks
   - Used for mapping categories to QuickBooks accounts
   - Cache results for 24 hours

4. createVendorIfNotExists(vendorName: string, userId: string): Promise<string>
   - Check if vendor exists in QuickBooks
   - Create new vendor if not found
   - Return vendor ID

MAPPING:
- Category "fuel" → QuickBooks Account "Fuel Expense"
- Category "tolls" → "Travel Expense"
- Category "maintenance" → "Vehicle Maintenance"
- Allow user to customize mappings in settings

ERROR HANDLING:
- Token expired → auto-refresh
- Rate limit hit → retry with exponential backoff
- Duplicate expense → skip with warning
- Invalid account → use default "Other Expense"

Provide complete sync service with error handling.
```

### LLM Prompt for QuickBooks Settings Page:

```
Create QuickBooks integration settings page:

FILE: /src/app/dashboard/settings/integrations/page.tsx

SECTIONS:

1. Connection Status
   - If not connected:
     - "Connect to QuickBooks" button
     - Benefits list:
       ✓ Auto-sync expenses
       ✓ Eliminate manual entry
       ✓ Match vendors automatically
   - If connected:
     - Company name
     - Connected date
     - Last sync timestamp
     - "Disconnect" button

2. Sync Settings
   - Auto-sync toggle (on/off)
   - Sync frequency dropdown:
     - Manual only
     - Every upload (immediate)
     - Daily (batch)
     - Weekly
   
3. Account Mapping
   - Table: FiscalNinja Category → QuickBooks Account
   - Rows:
     - Fuel → (dropdown of QB accounts)
     - Tolls → (dropdown)
     - Maintenance → (dropdown)
     - Insurance → (dropdown)
     - Other → (dropdown)
   - "Save Mappings" button

4. Sync History
   - Table of recent syncs:
     - Date/Time
     - Receipts synced
     - Status (Success, Failed, Partial)
     - View errors button
   - "Sync Now" button (manual trigger)

Use shadcn/ui components.

Provide complete settings page code.
```

### Validation Checklist:
- [ ] QuickBooks OAuth flow works
- [ ] Can connect to QuickBooks company
- [ ] Access token refreshes automatically
- [ ] Receipts sync to QuickBooks as expenses
- [ ] Vendors created automatically if not exist
- [ ] Category mapping works
- [ ] Batch sync works (10+ receipts at once)
- [ ] Sync errors handled gracefully
- [ ] Settings page shows connection status
- [ ] Can disconnect QuickBooks

---

## Step 3.3: Build Reporting Dashboard

**Objective:** Visualize expense data with charts and insights.

**Time Estimate:** 4 hours

### LLM Prompt for Reports Page:

```
Create a comprehensive reports dashboard:

FILE: /src/app/dashboard/reports/page.tsx

SECTIONS:

1. DATE RANGE SELECTOR (Top bar)
   - Presets: This Week, This Month, Last Month, This Quarter, This Year, Custom
   - Date picker for custom range

2. KEY METRICS CARDS
   - Total Expenses (large number)
   - Total Receipts (count)
   - Average Expense
   - Most Expensive Category
   - Top Spending Driver
   - Top Spending Truck

3. CHARTS (Use recharts library)

   Chart 1: Expenses Over Time (Line chart)
   - X-axis: Date
   - Y-axis: Amount
   - Group by: Day, Week, Month (toggle)
   - Show trend line

   Chart 2: Expenses by Category (Pie chart)
   - Slices: Fuel, Tolls, Maintenance, Insurance, Other
   - Show percentage and dollar amount
   - Click slice → filter receipts

   Chart 3: Expenses by Driver (Bar chart)
   - X-axis: Driver names
   - Y-axis: Total spent
   - Sortable

   Chart 4: Expenses by Truck (Bar chart)
   - X-axis: Truck numbers
   - Y-axis: Total spent
   - Identify high-cost trucks

4. TAX DEDUCTION SUMMARY
   - Total tax-deductible expenses
   - Breakdown by category
   - Estimated tax savings (if user enters tax bracket)
   - "Export for Tax Filing" button

5. DRIVER PERFORMANCE TABLE
   - Columns:
     - Driver Name
     - Total Receipts
     - Total Spent
     - Average per Receipt
     - Most Common Category
   - Sortable columns
   - Export to CSV

6. INSIGHTS (AI-generated or rule-based)
   - "Fuel expenses up 15% vs last month"
   - "Driver John has highest average receipt ($145)"
   - "Truck 203 maintenance costs above average"
   - "Most receipts submitted on Fridays"

DATA FETCHING:
- Use React Server Components
- Aggregate queries in database (faster than client-side)
- Cache results for 5 minutes

UI:
- Use shadcn/ui Card, Select
- Use recharts for charts
- Responsive (stack charts on mobile)
- Loading skeletons

Provide complete reports page with charts.
```

### LLM Prompt for Report Export:

```
Create a PDF report generator:

FILE: /src/lib/export/pdf-report.ts

FUNCTION: generatePDFReport(userId: string, dateRange: DateRange): Promise<Buffer>

REPORT STRUCTURE:

Page 1: Cover Page
- Company name (from profile)
- Report title: "Expense Report"
- Date range
- Generated date
- Logo (if uploaded)

Page 2: Executive Summary
- Total expenses
- Receipt count
- Top 3 categories
- Top 3 drivers
- Key insights

Page 3-4: Detailed Tables
- All receipts in date order
- Grouped by category
- Subtotals for each group
- Grand total

Page 5: Charts
- Expenses by category (pie chart)
- Expenses over time (line chart)
- Top drivers (bar chart)

Page 6: Tax Summary
- Tax-deductible expenses by category
- Total tax paid (sum of tax_amount field)
- Estimated tax savings

Use puppeteer or react-pdf library.

Provide complete PDF generator implementation.
```

### Validation Checklist:
- [ ] Reports page displays key metrics
- [ ] Line chart shows expenses over time
- [ ] Pie chart shows expenses by category
- [ ] Bar charts show driver/truck breakdowns
- [ ] Tax summary calculates correctly
- [ ] Driver performance table sorts properly
- [ ] Date range filter works
- [ ] PDF report generates successfully
- [ ] Charts render on mobile
- [ ] Data updates in real-time

---

## Step 3.4: Email Notifications

**Objective:** Send automated emails for important events.

**Time Estimate:** 2 hours

### LLM Prompt for Email Service:

```
Create an email notification service using Resend or SendGrid:

FILE: /src/lib/email/notifications.ts

EMAIL TEMPLATES:

1. Weekly Expense Summary
   TRIGGER: Every Sunday at 8pm
   TO: User email
   SUBJECT: "Your weekly expense report - {count} receipts, ${total}"
   CONTENT:
   - Total receipts uploaded this week
   - Total expenses
   - Breakdown by category
   - Receipts needing review (if any)
   - "View Full Report" button → dashboard

2. Receipt Needs Review Alert
   TRIGGER: When OCR confidence < 80%
   TO: User email
   SUBJECT: "Receipt needs review - {vendor}"
   CONTENT:
   - Receipt image thumbnail
   - Extracted data (with low-confidence fields highlighted)
   - "Review Now" button → review page

3. Monthly Report Ready
   TRIGGER: 1st of every month
   TO: User email
   SUBJECT: "Your {Month} expense report is ready"
   CONTENT:
   - Monthly summary
   - PDF attachment (auto-generated)
   - "Download Excel" link

4. QuickBooks Sync Failed
   TRIGGER: When sync error occurs
   TO: User email
   SUBJECT: "QuickBooks sync failed - action required"
   CONTENT:
   - Error description
   - How to fix (e.g., "Reconnect QuickBooks")
   - "Fix Now" button

5. Subscription Expiring
   TRIGGER: 3 days before subscription ends
   TO: User email
   SUBJECT: "Your FiscalNinja subscription expires in 3 days"
   CONTENT:
   - Renewal date
   - Current plan
   - "Renew Now" button

IMPLEMENTATION:
- Use Resend API (npm install resend)
- Store email templates in /src/lib/email/templates/
- Use React Email for template design
- Track email opens/clicks (optional)
- Handle bounces and unsubscribes

FUNCTIONS:
- sendWeeklySummary(userId: string): Promise<void>
- sendReviewAlert(receiptId: string): Promise<void>
- sendMonthlyReport(userId: string): Promise<void>
- sendSyncError(userId: string, error: string): Promise<void>

Use Supabase Edge Functions or Next.js API cron jobs for scheduling.

Provide complete email service implementation.
```

### LLM Prompt for Email Templates (React Email):

```
Create email templates using React Email:

FILE: /src/lib/email/templates/WeeklySummary.tsx

Use @react-email components:
- Html, Head, Body, Container
- Section, Row, Column
- Text, Heading, Button
- Img (for logo)

DESIGN:
- Clean, professional layout
- FiscalNinja branding colors
- Mobile-responsive
- Inline CSS (email-safe)
- Alt text for images
- Unsubscribe link in footer

Provide complete template for Weekly Summary email.
```

### Validation Checklist:
- [ ] Weekly summary email sends correctly
- [ ] Review alert triggers on low-confidence receipts
- [ ] Monthly report includes PDF attachment
- [ ] Sync error emails sent when QuickBooks fails
- [ ] Email templates render properly in Gmail/Outlook
- [ ] Unsubscribe link works
- [ ] Email scheduling works (cron jobs)

---

## Summary: Phase 3 Complete ✅

**What You've Built:**
- ✅ Excel export with summary sheets
- ✅ CSV export for spreadsheet users
- ✅ QuickBooks Online integration
- ✅ Comprehensive reporting dashboard with charts
- ✅ PDF report generator
- ✅ Automated email notifications

**Ready for Phase 4:** Multi-User & Subscriptions

---

## 🧪 Testing Phase 3

### Test Checklist:

**Excel Export:**
1. [ ] Export 10 receipts to Excel
2. [ ] Verify all data present (date, vendor, amount, etc.)
3. [ ] Summary sheet calculates totals correctly
4. [ ] Grouping by category works
5. [ ] CSV export works
6. [ ] QuickBooks CSV format valid

**QuickBooks Integration:**
1. [ ] OAuth connection works
2. [ ] Receipt syncs to QuickBooks
3. [ ] Vendor auto-created if not exists
4. [ ] Category mapping works
5. [ ] Token refresh works
6. [ ] Batch sync works
7. [ ] Disconnect works

**Reports:**
1. [ ] All charts render correctly
2. [ ] Date range filter works
3. [ ] Tax summary calculates properly
4. [ ] PDF report generates
5. [ ] Driver performance table accurate

**Emails:**
1. [ ] Weekly summary sends
2. [ ] Review alerts trigger
3. [ ] Templates render in Gmail
4. [ ] Unsubscribe works

# PHASE 4: Subscriptions & Multi-User (Days 15-21)

## Step 4.1: Implement Stripe Subscription Billing

**Objective:** Set up recurring subscription payments with Stripe.

**Time Estimate:** 4 hours

### LLM Prompt for Stripe Setup:

```
Create Stripe subscription billing integration:

FILE: /src/lib/payments/stripe.ts

REQUIREMENTS:

1. Stripe Products & Prices:
   Create these products in Stripe Dashboard:
   
   Product: FiscalNinja Solo
   - Price: $29/month
   - Description: "1-5 trucks, 200 receipts/month"
   - Price ID: price_solo_monthly
   
   Product: FiscalNinja Fleet
   - Price: $79/month
   - Description: "6-25 trucks, 1,000 receipts/month"
   - Price ID: price_fleet_monthly
   
   Product: FiscalNinja Enterprise
   - Price: $149/month
   - Description: "26-100 trucks, unlimited receipts"
   - Price ID: price_enterprise_monthly

2. Functions needed:

   a. createCheckoutSession(userId: string, priceId: string): Promise<string>
      - Create Stripe Checkout session
      - Success URL: /dashboard?session_id={CHECKOUT_SESSION_ID}
      - Cancel URL: /pricing
      - Customer email: pre-filled from user profile
      - Metadata: userId, planTier
      - Return session URL for redirect

   b. createPortalSession(customerId: string): Promise<string>
      - Create Stripe Customer Portal session
      - Allows users to:
        ✓ Update payment method
        ✓ View invoices
        ✓ Cancel subscription
        ✓ Upgrade/downgrade plan
      - Return portal URL

   c. handleWebhook(event: Stripe.Event): Promise<void>
      - Handle Stripe webhooks:
        
        checkout.session.completed:
        → Update user profile (subscription_status = 'active', stripe_customer_id)
        → Send welcome email
        
        invoice.payment_succeeded:
        → Extend subscription period
        → Send receipt email
        
        invoice.payment_failed:
        → Update status to 'past_due'
        → Send payment failed email
        → Retry payment 3 times over 10 days
        
        customer.subscription.deleted:
        → Update status to 'cancelled'
        → Downgrade features
        → Send cancellation email

   d. getSubscriptionStatus(userId: string): Promise<SubscriptionStatus>
      - Fetch current subscription from Stripe
      - Return: plan tier, status, current_period_end, cancel_at_period_end

3. Webhook Endpoint:

   FILE: /src/app/api/webhooks/stripe/route.ts
   
   ENDPOINT: POST /api/webhooks/stripe
   
   - Verify webhook signature (Stripe secret)
   - Parse event type
   - Call handleWebhook()
   - Return 200 OK

4. Usage Tracking:

   - Track receipts uploaded each month
   - If user exceeds tier limit:
     → Block new uploads
     → Show upgrade modal
     → Send email: "You've reached your limit - upgrade to continue"

Use Stripe SDK (npm install stripe @stripe/stripe-js).

Provide complete Stripe integration with webhook handling.
```

### LLM Prompt for Pricing Page:

```
Create a pricing page with subscription tiers:

FILE: /src/app/pricing/page.tsx

LAYOUT:

Hero Section:
- Headline: "Simple, Transparent Pricing"
- Subheadline: "Choose the plan that fits your fleet size"
- Toggle: Monthly / Annual (10% discount)

Pricing Cards (3 columns):

CARD 1: SOLO
- Price: $29/month
- "For owner-operators"
- Features:
  ✓ 1-5 trucks
  ✓ 200 receipts/month
  ✓ OCR processing
  ✓ Excel export
  ✓ Email support
  ✓ Mobile app
- Button: "Start Free Trial" (14 days)

CARD 2: FLEET ⭐ POPULAR
- Price: $79/month
- "For growing fleets"
- Badge: "Most Popular"
- Features:
  ✓ 6-25 trucks
  ✓ 1,000 receipts/month
  ✓ QuickBooks integration
  ✓ Multi-user access
  ✓ Priority support
  ✓ Custom categories
  ✓ Advanced reports
- Button: "Start Free Trial"

CARD 3: ENTERPRISE
- Price: $149/month
- "For large operations"
- Features:
  ✓ 26-100 trucks
  ✓ Unlimited receipts
  ✓ API access
  ✓ White-label option
  ✓ Dedicated account manager
  ✓ Custom integrations
  ✓ Phone support
- Button: "Start Free Trial"

FAQ Section:
- "Can I change plans?" → Yes, upgrade/downgrade anytime
- "What happens after trial?" → Automatically charged unless you cancel
- "Do you offer refunds?" → 30-day money-back guarantee
- "What payment methods?" → Credit card, debit card
- "Is there a setup fee?" → No setup fees

Comparison Table (below cards):
- Feature-by-feature comparison
- All plans listed side-by-side

Testimonials (optional):
- 2-3 customer quotes

CTA Section:
- "Start your 14-day free trial today"
- "No credit card required"

IMPLEMENTATION:
- Click "Start Free Trial" → redirect to Stripe Checkout
- Pre-select plan based on button clicked
- Use shadcn/ui Card, Button, Badge
- Responsive design (stack cards on mobile)

Provide complete pricing page code.
```

### LLM Prompt for Subscription Settings:

```
Create subscription management page:

FILE: /src/app/dashboard/settings/subscription/page.tsx

SECTIONS:

1. Current Plan
   - Plan name (Solo, Fleet, Enterprise)
   - Status badge (Active, Past Due, Cancelled)
   - Billing cycle (Monthly/Annual)
   - Next billing date
   - "Manage Subscription" button → Stripe Customer Portal

2. Usage This Month
   - Progress bar: X / 200 receipts used
   - Breakdown:
     - Receipts uploaded: 47
     - Trucks active: 3
     - Drivers active: 5
   - Warning if near limit: "You're at 90% capacity - consider upgrading"

3. Upgrade Options (if not on Enterprise)
   - Show next tier up
   - Benefits of upgrading
   - "Upgrade Now" button → Stripe Checkout (immediate upgrade)

4. Billing History
   - Table of invoices:
     - Date
     - Amount
     - Status (Paid, Failed)
     - Download button (PDF invoice)
   - Fetched from Stripe API

5. Cancel Subscription
   - "Cancel Subscription" button (with confirmation modal)
   - Show retention offer: "Get 20% off for 3 months if you stay"
   - If cancelled, show: "Access until {end_date}"

Use Stripe SDK to fetch data.

Provide complete subscription settings page.
```

### Validation Checklist:
- [ ] Stripe Checkout flow works
- [ ] Payment successful → subscription activated
- [ ] Webhook updates database correctly
- [ ] Customer Portal allows plan changes
- [ ] Failed payment triggers retry logic
- [ ] Cancellation works correctly
- [ ] Usage tracking shows accurate counts
- [ ] Upgrade/downgrade works
- [ ] Invoice downloads work
- [ ] 14-day free trial works

---

## Step 4.2: Multi-User Support

**Objective:** Allow office managers and drivers to access the app with role-based permissions.

**Time Estimate:** 5 hours

### LLM Prompt for User Roles System:

```
Implement role-based access control (RBAC):

DATABASE UPDATES:

1. Add to profiles table:
   - role (enum: 'owner', 'manager', 'driver')
   - parent_user_id (UUID, nullable, references profiles.id)
     → For manager/driver accounts, points to owner

2. New table: team_members
   - id (UUID, primary key)
   - owner_id (UUID, references profiles.id)
   - user_id (UUID, references profiles.id)
   - role (enum: 'owner', 'manager', 'driver')
   - invited_email (text)
   - invited_at (timestamp)
   - accepted_at (timestamp, nullable)
   - active (boolean, default true)
   - permissions (JSONB, e.g., {"canDelete": false, "canExport": true})

ROLES & PERMISSIONS:

OWNER:
- Full access to everything
- Can invite/remove users
- View all receipts
- Manage subscription
- Export reports
- Delete receipts
- Configure integrations

MANAGER:
- View all receipts
- Upload receipts
- Edit/review receipts
- Export reports
- Cannot manage subscription
- Cannot delete users
- Cannot see billing info

DRIVER:
- Upload receipts only
- View own receipts
- Cannot see other drivers' receipts
- Cannot export
- Cannot edit expense categories
- Read-only dashboard

IMPLEMENTATION:

FILE: /src/lib/auth/permissions.ts

```typescript
type Permission = 
  | 'receipts.view.all'
  | 'receipts.view.own'
  | 'receipts.upload'
  | 'receipts.edit'
  | 'receipts.delete'
  | 'reports.view'
  | 'reports.export'
  | 'team.manage'
  | 'settings.view'
  | 'subscription.manage';

const rolePermissions: Record<UserRole, Permission[]> = {
  owner: [/* all permissions */],
  manager: [/* subset */],
  driver: [/* minimal */],
};

function hasPermission(user: User, permission: Permission): boolean {
  return rolePermissions[user.role].includes(permission);
}

function requirePermission(permission: Permission) {
  // Middleware for API routes
  // Returns 403 if user lacks permission
}
```

Use in API routes:
```typescript
// Example: Delete receipt
export async function DELETE(req: Request) {
  const user = await getCurrentUser(req);
  if (!hasPermission(user, 'receipts.delete')) {
    return new Response('Forbidden', { status: 403 });
  }
  // ... proceed with delete
}
```

Provide complete RBAC implementation with middleware.
```

### LLM Prompt for Team Management UI:

```
Create team management interface:

FILE: /src/app/dashboard/settings/team/page.tsx

SECTIONS:

1. Team Members List
   - Table columns:
     - Name
     - Email
     - Role (badge)
     - Status (Active, Pending Invite)
     - Last Active
     - Actions (Edit, Remove)
   
2. Invite User Form
   - Email input
   - Role selector (Manager, Driver)
   - Optional: Assign to specific truck (for drivers)
   - "Send Invite" button
   
   WORKFLOW:
   - Click "Send Invite"
   - Create team_members record (invited_at = now, accepted_at = null)
   - Send email with invite link:
     https://fiscalninja.com/accept-invite/{token}
   - Recipient clicks link → create account → automatically added to team

3. Pending Invites
   - List of invites not yet accepted
   - "Resend Invite" button
   - "Cancel Invite" button

4. Role Permissions Guide
   - Table showing what each role can do:
     
     | Permission | Owner | Manager | Driver |
     |------------|-------|---------|--------|
     | Upload receipts | ✓ | ✓ | ✓ |
     | View all receipts | ✓ | ✓ | ✗ |
     | Edit receipts | ✓ | ✓ | ✗ |
     | Delete receipts | ✓ | ✗ | ✗ |
     | Export reports | ✓ | ✓ | ✗ |
     | Manage team | ✓ | ✗ | ✗ |
     | Manage billing | ✓ | ✗ | ✗ |

ONLY OWNERS can access this page.

Use shadcn/ui Table, Dialog, Select, Button.

Provide complete team management page.
```

### LLM Prompt for Driver Mobile View:

```
Create simplified driver interface:

FILE: /src/app/driver/page.tsx

PURPOSE: Streamlined UI for drivers (mobile-first)

FEATURES:

1. Quick Upload Button (Large, center of screen)
   - "Upload Receipt" button
   - Opens camera on mobile
   - Shows recent uploads below (last 10)

2. My Receipts List
   - Only show receipts uploaded by this driver
   - Simple card layout:
     - Date
     - Vendor
     - Amount
     - Thumbnail
   - No edit/delete options (read-only)

3. Simple Stats
   - "Receipts this week: 12"
   - "Total this month: $1,234.56"

NO ACCESS TO:
- Other drivers' receipts
- Full dashboard
- Reports
- Settings
- Team management

NAVIGATION:
- Minimal nav: Home, Upload, My Receipts, Logout
- No complex menus

Mobile-optimized design (large buttons, touch-friendly).

Provide complete driver interface.
```

### Validation Checklist:
- [ ] Owners can invite managers/drivers
- [ ] Invite email sent with link
- [ ] Invitee can create account and join team
- [ ] Role permissions enforced (drivers can't delete)
- [ ] Managers see all receipts
- [ ] Drivers only see own receipts
- [ ] API routes check permissions
- [ ] Team list shows all members
- [ ] Can remove team members
- [ ] Driver mobile view works

---

## Step 4.3: Usage Limits & Overage Handling

**Objective:** Enforce subscription tier limits and handle overages gracefully.

**Time Estimate:** 2 hours

### LLM Prompt for Usage Tracking:

```
Implement usage tracking and enforcement:

FILE: /src/lib/subscriptions/usage.ts

FUNCTIONS:

1. checkReceiptLimit(userId: string): Promise<UsageStatus>
   
   LOGIC:
   - Get user's subscription tier
   - Count receipts uploaded this month
   - Compare to tier limit:
     - Solo: 200/month
     - Fleet: 1,000/month
     - Enterprise: unlimited
   
   RETURN:
   ```typescript
   interface UsageStatus {
     allowed: boolean;
     used: number;
     limit: number;
     percentage: number; // 0-100
     message?: string;
   }
   ```

2. enforceLimit(userId: string): Promise<void>
   - Called before each upload
   - If limit exceeded:
     → Throw error: "You've reached your monthly limit"
     → Show upgrade modal
     → Send email notification
   
3. getUsageStats(userId: string): Promise<UsageStats>
   - Receipts this month
   - Trucks active
   - Drivers active
   - Storage used (MB)
   - Return breakdown by category

4. resetMonthlyUsage(userId: string): Promise<void>
   - Called on 1st of month (cron job)
   - Reset monthly counters
   - Send usage summary email

MIDDLEWARE:
- Add to upload API route:
  ```typescript
  const usage = await checkReceiptLimit(userId);
  if (!usage.allowed) {
    return new Response('Limit exceeded', { status: 402 });
  }
  ```

Provide complete usage tracking implementation.
```

### LLM Prompt for Upgrade Modal:

```
Create upgrade prompt modal:

FILE: /src/components/modals/UpgradeModal.tsx

TRIGGER: When user hits usage limit

CONTENT:

Headline: "You've reached your plan limit"

Body:
- "You've uploaded 200/200 receipts this month"
- "Upgrade to Fleet plan for 1,000 receipts/month"

Benefits of upgrading:
✓ 5x more receipts (1,000/month)
✓ QuickBooks integration
✓ Multi-user access
✓ Priority support

Pricing comparison:
- Current: Solo ($29/month)
- Upgrade to: Fleet ($79/month)
- Difference: +$50/month

Buttons:
- "Upgrade Now" (primary) → Stripe Checkout
- "Remind Me Later" (secondary)
- "Learn More" → /pricing

ALTERNATE FLOW:
If user is on annual plan, show:
- "We'll prorate your current plan"
- "You'll pay $X today for the remaining Y days"

Use shadcn/ui Dialog, Button.

Provide complete modal code.
```

### Validation Checklist:
- [ ] Usage correctly tracks receipts/month
- [ ] Upload blocked when limit reached
- [ ] Upgrade modal shows correct pricing
- [ ] Upgrade flow works (immediate upgrade)
- [ ] Proration calculated correctly
- [ ] Usage resets on 1st of month
- [ ] Usage stats accurate
- [ ] Warning shown at 90% capacity

---

## Summary: Phase 4 Complete ✅

**What You've Built:**
- ✅ Stripe subscription billing (3 tiers)
- ✅ Webhook handling for payment events
- ✅ Customer Portal integration
- ✅ Role-based access control (Owner, Manager, Driver)
- ✅ Team member invitations
- ✅ Usage tracking and limit enforcement
- ✅ Upgrade prompts and modal
- ✅ Driver-specific mobile interface

**Ready for Phase 5:** Polish & Launch

---

## 🧪 Testing Phase 4

### Test Checklist:

**Subscriptions:**
1. [ ] Complete checkout for Solo plan
2. [ ] Subscription activated in database
3. [ ] Webhook updates subscription status
4. [ ] Failed payment triggers retry
5. [ ] Cancellation works correctly
6. [ ] Upgrade Solo → Fleet works
7. [ ] Downgrade Fleet → Solo works
8. [ ] Annual billing discount applied
9. [ ] Free trial works (14 days)
10. [ ] Invoice emails sent

**Multi-User:**
1. [ ] Owner can invite manager
2. [ ] Manager receives invite email
3. [ ] Manager creates account and joins team
4. [ ] Manager sees all receipts
5. [ ] Manager cannot access billing
6. [ ] Driver invited and accepted
7. [ ] Driver only sees own receipts
8. [ ] Driver cannot delete receipts
9. [ ] Permissions enforced in API
10. [ ] Team list accurate

**Usage Limits:**
1. [ ] Upload blocked at 200 receipts (Solo)
2. [ ] Upgrade modal shown
3. [ ] Usage stats accurate
4. [ ] Monthly reset works
5. [ ] Overage email sent
6. [ ] Upgrade increases limit immediately

# PHASE 5: Polish, Testing & Launch (Days 22-30)

## Step 5.1: Build Marketing Landing Page

**Objective:** Create compelling landing page to convert visitors to trial users.

**Time Estimate:** 4 hours

### LLM Prompt for Landing Page:

```
Create a high-converting SaaS landing page:

FILE: /src/app/page.tsx (replaces default homepage)

STRUCTURE:

=== SECTION 1: HERO ===
Above the fold:
- Headline: "Stop Wasting 20 Hours a Month on Receipt Entry"
- Subheadline: "Truck drivers snap photos. Our AI extracts the data. You download organized expense reports."
- CTA Button: "Start Free Trial" (large, prominent)
- Visual: Screenshot or demo GIF showing:
  1. Driver photo of receipt
  2. App extracting data
  3. Excel file with organized expenses

=== SECTION 2: PROBLEM ===
"The Receipt Chaos Trucking Companies Face"
- 3 pain points with icons:
  1. 📸 Drivers text blurry receipt photos via WhatsApp
  2. ⌨️ Owners spend 20+ hours/month typing data
  3. 💸 Lost receipts = $3,000-$5,000 in unclaimed deductions

=== SECTION 3: SOLUTION ===
"How FiscalNinja Works"
3-step process with visuals:
1. Upload: "Drivers snap photos (mobile-friendly)"
2. AI Extracts: "Our OCR reads vendor, amount, date automatically"
3. Export: "Download Excel or sync to QuickBooks"

=== SECTION 4: BENEFITS ===
"Save Time. Save Money. Stay Organized."

Grid of benefits (icons + text):
✅ Save 20+ hours/month (no manual entry)
✅ Never lose a tax deduction (all receipts organized)
✅ Faster driver reimbursements (happy drivers = lower turnover)
✅ QuickBooks integration (seamless accounting)
✅ Real-time reports (know your expenses instantly)
✅ Mobile-first (works on any smartphone)

=== SECTION 5: SOCIAL PROOF ===
"Trusted by Trucking Companies Across North America"

Testimonials (3 cards):
1. "Saved me 25 hours a month. Game-changer." - John M., Owner (12 trucks)
2. "Drivers love the app. No more lost receipts." - Sarah K., Office Manager
3. "Found $4,000 in missed deductions." - Mike D., Solo operator

Stats (if available):
- "1,000+ receipts processed"
- "500+ hours saved"
- "$50,000+ in tax deductions recovered"

=== SECTION 6: FEATURES ===
"Everything You Need in One Platform"

Feature grid (6 items with icons):
1. 📸 Mobile Upload - Camera-friendly interface
2. 🤖 AI OCR - 90%+ accuracy on receipts
3. 📊 Excel Export - Organized expense reports
4. 💼 QuickBooks Sync - Automatic expense creation
5. 👥 Multi-User - Drivers, managers, owners
6. 📈 Reports - Visualize spending trends

=== SECTION 7: PRICING PREVIEW ===
"Simple Pricing for Every Fleet Size"

3 pricing cards (simplified):
- Solo: $29/month - "1-5 trucks"
- Fleet: $79/month - "6-25 trucks" (MOST POPULAR)
- Enterprise: $149/month - "26-100 trucks"

Button: "See All Plans" → /pricing

=== SECTION 8: FAQ ===
"Frequently Asked Questions"

Q&A accordion:
1. "How accurate is the OCR?" 
   → 90-95% for standard receipts. You can review/edit before exporting.
2. "Can drivers use their own phones?"
   → Yes! Works on any smartphone with a camera.
3. "Does it work with QuickBooks?"
   → Yes, automatic sync to QuickBooks Online.
4. "What if I have >100 trucks?"
   → Contact us for enterprise pricing.
5. "Is there a free trial?"
   → Yes, 14 days free. No credit card required.

=== SECTION 9: FINAL CTA ===
"Ready to Save 20 Hours a Month?"

- Big CTA button: "Start Your Free Trial"
- Subtext: "14-day free trial • No credit card required • Cancel anytime"
- Badges: "✓ 90-day money-back guarantee" "✓ GDPR compliant"

=== FOOTER ===
- Links: Pricing, Features, About, Contact, Terms, Privacy
- Social: Twitter, LinkedIn
- Copyright

DESIGN REQUIREMENTS:
- Use Tailwind CSS with your brand colors
- Mobile-responsive (stack sections on mobile)
- Fast loading (<2 seconds)
- Optimized images (WebP format)
- Clear CTAs on every section
- Use shadcn/ui Button, Card, Accordion
- Add animations (scroll-triggered fade-ins)

Provide complete landing page code.
```

### LLM Prompt for Demo Video Script:

```
Create a 60-second demo video script:

SCENE 1 (0-10s): Problem
Visual: Frustrated owner at desk with pile of receipts
Voiceover: "Tired of spending 20 hours a month typing receipt data?"

SCENE 2 (10-20s): Solution Introduction
Visual: FiscalNinja logo animation
VO: "Meet FiscalNinja - the automated receipt manager for trucking companies."

SCENE 3 (20-35s): How It Works
Visual: Split-screen demo
- Left: Driver taking photo of fuel receipt on phone
- Right: App screen showing OCR extracting data
VO: "Drivers snap photos. Our AI reads vendor, amount, and date. Everything organized automatically."

SCENE 4 (35-50s): Results
Visual: Dashboard showing reports, then Excel file download
VO: "Export to Excel or sync to QuickBooks. Never lose a tax deduction again."

SCENE 5 (50-60s): CTA
Visual: Pricing page
VO: "Start your free trial today. No credit card required."
Text on screen: "fiscalninja.com"

PRODUCTION TIPS:
- Use screen recording for app demo (Loom or ScreenFlow)
- Stock footage for driver scenes (Pexels, Unsplash)
- Add captions (80% watch on mute)
- Background music (upbeat, corporate)
- Export in multiple formats (YouTube, Twitter, LinkedIn)

You can create this yourself or hire on Fiverr ($50-$150).
```

### Validation Checklist:
- [ ] Landing page loads fast (<2s)
- [ ] All CTAs work (redirect to signup)
- [ ] Mobile responsive
- [ ] FAQ accordion works
- [ ] Testimonials display correctly
- [ ] Pricing preview accurate
- [ ] Demo video embedded (if created)
- [ ] SEO meta tags added

---

## Step 5.2: SEO & Analytics Setup

**Objective:** Optimize for search engines and track user behavior.

**Time Estimate:** 2 hours

### LLM Prompt for SEO Optimization:

```
Optimize FiscalNinja for SEO:

TASKS:

1. Meta Tags (for all pages)
   
   FILE: /src/app/layout.tsx
   
   Add:
   ```typescript
   export const metadata: Metadata = {
     title: 'FiscalNinja - Automated Receipt Manager for Trucking Companies',
     description: 'Stop wasting 20 hours/month on receipt entry. AI-powered OCR extracts data from fuel receipts. Export to Excel or QuickBooks. Free 14-day trial.',
     keywords: 'trucking receipts, fuel receipt tracker, expense management, QuickBooks integration, OCR receipt scanner',
     openGraph: {
       title: 'FiscalNinja - Save 20 Hours/Month on Receipt Entry',
       description: 'Automated receipt management for trucking companies',
       images: ['/og-image.png'],
       type: 'website',
     },
     twitter: {
       card: 'summary_large_image',
       title: 'FiscalNinja - Automated Receipt Manager',
       description: 'AI-powered receipt tracking for trucking companies',
       images: ['/twitter-card.png'],
     },
   };
   ```

2. Structured Data (Schema.org)
   
   Add to homepage:
   ```typescript
   const jsonLd = {
     '@context': 'https://schema.org',
     '@type': 'SoftwareApplication',
     name: 'FiscalNinja',
     applicationCategory: 'BusinessApplication',
     offers: {
       '@type': 'Offer',
       price: '29',
       priceCurrency: 'USD',
     },
     aggregateRating: {
       '@type': 'AggregateRating',
       ratingValue: '4.8',
       ratingCount: '127',
     },
   };
   ```

3. Sitemap Generation
   
   FILE: /src/app/sitemap.ts
   
   Generate sitemap.xml with:
   - Homepage
   - Pricing
   - Features
   - Blog posts (if added)

4. robots.txt
   
   FILE: /public/robots.txt
   
   ```
   User-agent: *
   Allow: /
   Sitemap: https://fiscalninja.com/sitemap.xml
   ```

5. Page Speed Optimization
   - Compress images (use next/image with optimization)
   - Lazy load below-the-fold content
   - Minimize JavaScript bundles
   - Enable caching headers

6. Core Web Vitals
   - LCP (Largest Contentful Paint) < 2.5s
   - FID (First Input Delay) < 100ms
   - CLS (Cumulative Layout Shift) < 0.1

TARGET KEYWORDS:
- "trucking receipt management"
- "fuel receipt tracker"
- "expense tracking for truckers"
- "QuickBooks receipt scanner"
- "automated receipt entry"

Create blog posts targeting these keywords (optional for later).

Provide SEO optimization checklist and code.
```

### LLM Prompt for Analytics Setup:

```
Set up analytics tracking:

TOOLS:

1. Vercel Analytics (free)
   - Add to /src/app/layout.tsx:
     ```typescript
     import { Analytics } from '@vercel/analytics/react';
     
     export default function RootLayout({ children }) {
       return (
         <html>
           <body>
             {children}
             <Analytics />
           </body>
         </html>
       );
     }
     ```

2. Mixpanel (free tier)
   - Track custom events:
     - User signed up
     - Receipt uploaded
     - Export generated
     - Subscription created
     - Upgrade triggered
   
   FILE: /src/lib/analytics/mixpanel.ts
   
   ```typescript
   import mixpanel from 'mixpanel-browser';
   
   mixpanel.init('YOUR_TOKEN');
   
   export function trackEvent(event: string, properties?: object) {
     mixpanel.track(event, properties);
   }
   
   export function identifyUser(userId: string, traits: object) {
     mixpanel.identify(userId);
     mixpanel.people.set(traits);
   }
   ```
   
   Use in components:
   ```typescript
   // When user uploads receipt
   trackEvent('Receipt Uploaded', {
     category: 'fuel',
     confidence: 92,
   });
   ```

3. Google Analytics (optional)
   - Standard pageview tracking
   - Conversion tracking for signups

METRICS TO TRACK:

Acquisition:
- Traffic sources (Google, LinkedIn, direct)
- Landing page conversion rate
- Sign-up funnel drop-off

Activation:
- Time to first receipt upload
- Free trial start rate
- Onboarding completion %

Engagement:
- Receipts uploaded per user
- Dashboard sessions per week
- Feature usage (QuickBooks, Export)

Revenue:
- Trial-to-paid conversion
- Plan distribution (Solo/Fleet/Enterprise)
- MRR growth
- Churn rate

Retention:
- 30-day retention
- 90-day retention
- DAU/MAU ratio

Provide complete analytics implementation.
```

### Validation Checklist:
- [ ] Meta tags set on all pages
- [ ] Open Graph images display on social media
- [ ] Sitemap.xml generated
- [ ] robots.txt configured
- [ ] Google Search Console verified
- [ ] Vercel Analytics tracking pageviews
- [ ] Mixpanel tracking custom events
- [ ] Core Web Vitals pass (test with Lighthouse)
- [ ] Mobile SEO score >90
- [ ] Page speed <2s

---

## Step 5.3: Security Audit & Production Checklist

**Objective:** Ensure app is secure and production-ready.

**Time Estimate:** 3 hours

### Security Checklist:

```
SECURITY AUDIT TASKS:

1. Authentication & Authorization
   - [ ] Passwords hashed with bcrypt (minimum 10 rounds)
   - [ ] JWT tokens expire (1 hour)
   - [ ] Refresh tokens stored securely (httpOnly cookies)
   - [ ] Email verification required before login
   - [ ] Rate limiting on login endpoint (max 5 attempts/minute)
   - [ ] CSRF protection enabled
   - [ ] Session timeout after 24 hours of inactivity

2. Database Security
   - [ ] Row Level Security (RLS) enabled on all tables
   - [ ] Prepared statements (no SQL injection)
   - [ ] Sensitive data encrypted (Stripe keys, QuickBooks tokens)
   - [ ] Database backups automated (daily)
   - [ ] Connection pooling configured

3. API Security
   - [ ] All endpoints require authentication (except public pages)
   - [ ] API rate limiting (100 requests/minute per user)
   - [ ] Input validation on all forms (Zod schemas)
   - [ ] XSS protection (sanitize user input)
   - [ ] CORS configured (only allow fiscalninja.com)
   - [ ] Webhook signature verification (Stripe)

4. File Upload Security
   - [ ] File type validation (only images/PDFs)
   - [ ] File size limits enforced (10MB max)
   - [ ] Antivirus scanning (optional: use ClamAV)
   - [ ] Private file storage (Supabase Storage with auth)
   - [ ] Signed URLs for temporary access
   - [ ] No executable files allowed

5. Secrets Management
   - [ ] Environment variables in .env.local (not committed)
   - [ ] Production secrets in Vercel dashboard
   - [ ] API keys rotated regularly
   - [ ] No hardcoded secrets in code
   - [ ] Use Vercel Secret Scanner

6. HTTPS & Certificates
   - [ ] Force HTTPS on all pages
   - [ ] HSTS header enabled
   - [ ] SSL certificate valid (auto-renewed)
   - [ ] Secure cookies (secure, httpOnly, sameSite)

7. Third-Party Services
   - [ ] Stripe webhook secrets verified
   - [ ] Google Cloud credentials secured
   - [ ] QuickBooks OAuth tokens encrypted
   - [ ] Supabase RLS policies tested

8. Error Handling
   - [ ] No sensitive data in error messages
   - [ ] Custom error pages (404, 500)
   - [ ] Error logging to Sentry (not console)
   - [ ] Stack traces hidden in production

9. Compliance
   - [ ] Privacy policy published
   - [ ] Terms of service published
   - [ ] GDPR-compliant (data export, deletion)
   - [ ] Cookie consent banner (if using analytics cookies)
   - [ ] Data retention policy (delete old receipts?)

10. Monitoring & Alerts
    - [ ] Uptime monitoring (UptimeRobot or similar)
    - [ ] Error alerts (Sentry email notifications)
    - [ ] Failed payment alerts
    - [ ] Database performance monitoring
    - [ ] Unusual activity alerts (100+ uploads in 1 hour)

TEST WITH:
- OWASP ZAP (vulnerability scanner)
- npm audit (dependency vulnerabilities)
- Snyk (security scanning)
```

### LLM Prompt for Security Middleware:

```
Create security middleware for Next.js:

FILE: /src/middleware.ts

FEATURES:

1. Rate Limiting
   - Track requests per IP
   - Max 100 requests/minute
   - Return 429 Too Many Requests if exceeded

2. Authentication Check
   - Verify JWT token on protected routes
   - Redirect to /auth/login if unauthenticated
   - Refresh token if expired

3. CSRF Protection
   - Verify CSRF token on POST/PUT/DELETE
   - Generate token on page load

4. Security Headers
   - X-Frame-Options: DENY
   - X-Content-Type-Options: nosniff
   - Referrer-Policy: strict-origin-when-cross-origin
   - Content-Security-Policy: (strict CSP)

5. Input Sanitization
   - Strip HTML tags from text inputs
   - Validate email formats
   - Prevent XSS attacks

Use middleware on these routes:
- /dashboard/*
- /api/*

Provide complete middleware implementation.
```

### Production Deployment Checklist:

```
BEFORE DEPLOYING TO PRODUCTION:

Environment Setup:
- [ ] Domain purchased (fiscalninja.com)
- [ ] DNS configured (Vercel nameservers)
- [ ] SSL certificate issued
- [ ] Environment variables set in Vercel dashboard
- [ ] Database production instance created (Supabase Pro)
- [ ] Stripe live mode enabled (switch from test keys)

Code:
- [ ] All console.logs removed
- [ ] Debug flags disabled
- [ ] Error boundaries added
- [ ] Loading states implemented
- [ ] Empty states implemented
- [ ] TypeScript errors fixed (npm run build succeeds)
- [ ] ESLint warnings resolved

Testing:
- [ ] Manual testing completed (all features)
- [ ] Cross-browser testing (Chrome, Safari, Firefox)
- [ ] Mobile testing (iOS, Android)
- [ ] Load testing (simulate 100 concurrent users)
- [ ] OCR tested with 50+ receipt types

Performance:
- [ ] Images optimized (WebP, lazy loading)
- [ ] Code splitting enabled
- [ ] Database queries optimized (indexes added)
- [ ] Caching configured (Redis or in-memory)
- [ ] CDN configured (Vercel Edge Network)

Monitoring:
- [ ] Sentry error tracking enabled
- [ ] Vercel Analytics configured
- [ ] Uptime monitoring setup
- [ ] Slack/email alerts configured

Legal:
- [ ] Privacy policy reviewed by lawyer (optional)
- [ ] Terms of service finalized
- [ ] GDPR compliance verified
- [ ] PCI compliance (Stripe handles this)

Launch:
- [ ] Soft launch to 5 beta users
- [ ] Fix critical bugs
- [ ] Public launch announcement
- [ ] Monitor errors for first 48 hours
```

### Validation Checklist:
- [ ] Security audit completed
- [ ] All vulnerabilities patched
- [ ] Production environment configured
- [ ] SSL working correctly
- [ ] Error tracking active
- [ ] Monitoring alerts working
- [ ] Performance acceptable (Lighthouse score >90)
- [ ] No console errors on any page
- [ ] Cross-browser tested
- [ ] Mobile responsive verified

---

## Step 5.4: Customer Acquisition Plan

**Objective:** Get first 10 paying customers in 30 days.

**Time Estimate:** Ongoing effort

### LLM Prompt for Cold Email Templates:

```
Create cold email templates for trucking company outreach:

TEMPLATE 1: Pain-Point Focused

Subject: Still typing fuel receipts into Excel?

Hi [Name],

Quick question: How much time does your team spend managing fuel receipts each month?

Most trucking companies I talk to waste 15-25 hours/month:
• Drivers text blurry photos
• Office manager types data into spreadsheets
• Receipts get lost (missed tax deductions)

I built FiscalNinja to fix this. Drivers snap photos, our AI reads the receipt, you download an Excel file. 5 minutes/week instead of 20 hours/month.

Would you be open to a 15-minute demo? I'll show you exactly how it works with your receipts.

Best,
[Your Name]

P.S. We're offering a 30-day free trial (no credit card required).

---

TEMPLATE 2: Social Proof

Subject: How [Competitor] saved 25 hours/month

Hi [Name],

[Competitor Company] in [City] was spending 20+ hours/month organizing fuel receipts. They switched to FiscalNinja 3 months ago.

Results:
✓ Saved 25 hours/month (no more manual entry)
✓ Found $4,000 in missed tax deductions
✓ Drivers love the mobile app

I'd love to show you how we can do the same for [Company Name].

Free demo this week? Takes 15 minutes.

Best,
[Your Name]

---

TEMPLATE 3: QuickBooks Hook

Subject: Sync fuel receipts to QuickBooks automatically?

Hi [Name],

I noticed [Company Name] uses QuickBooks. Quick question: Do you manually enter fuel receipts into QB, or does your bookkeeper handle it?

Either way, it's probably taking 10-15 hours/month.

FiscalNinja syncs receipts directly to QuickBooks:
• Drivers upload photos
• AI extracts vendor, amount, date
• Expenses auto-created in QuickBooks
• Zero manual entry

Want to see it in action? I can demo with your actual receipts.

Best,
[Your Name]

---

FOLLOW-UP SEQUENCE:

Day 1: Send initial email
Day 4: Follow-up if no response:
  "Hi [Name], just bumping this to the top of your inbox. Still wasting time on receipt entry?"
Day 7: Final follow-up:
  "Last time I'll bug you! If receipt management isn't a pain point, no worries. If it is, we can save you 20 hours/month."

TARGETING:
- LinkedIn Sales Navigator: "Owner" + "Trucking" + [Your State]
- ZoomInfo: Get emails for trucking companies (1-50 trucks)
- Google: "trucking company [city]" + contact form
```

### Content Marketing Plan:

```
CONTENT STRATEGY:

1. Blog Posts (SEO-focused)
   
   Post 1: "How to Track Fuel Receipts for Tax Deductions (2026 Guide)"
   - Target keyword: "fuel receipt tracking"
   - 2,000 words, step-by-step guide
   - CTA: "Automate this with FiscalNinja"
   
   Post 2: "QuickBooks Receipt Scanner: 5 Tools Compared"
   - Target keyword: "QuickBooks receipt scanner"
   - Compare FiscalNinja vs. Expensify vs. Shoeboxed
   - CTA: Free trial
   
   Post 3: "Trucking Expense Categories for Tax Filing"
   - Target keyword: "trucking tax deductions"
   - List all deductible expenses
   - CTA: Track automatically
   
   Publish 2-4 posts/month.

2. YouTube Videos
   
   Video 1: "How to Use FiscalNinja (5-Minute Tutorial)"
   - Screen recording walkthrough
   - Upload, review, export
   
   Video 2: "Sync Fuel Receipts to QuickBooks Automatically"
   - QuickBooks integration demo
   
   Video 3: "5 Tax Deductions Truckers Miss (and How to Avoid It)"
   - Educational content
   - Mention FiscalNinja as solution
   
   Post 1-2 videos/month.

3. LinkedIn Posts
   
   Post every 2-3 days:
   - Tips for trucking bookkeeping
   - Customer success stories
   - Feature announcements
   - Industry news (fuel prices, regulations)
   
   Engage with trucking groups:
   - "Trucking Business Owners"
   - "Owner Operators Network"

4. Reddit
   
   Communities:
   - r/Truckers
   - r/smallbusiness
   - r/Entrepreneur
   
   Provide value (not spammy):
   - Answer questions about expense tracking
   - Share blog posts
   - Offer free advice
   
   Mention FiscalNinja only when relevant.

5. Partnerships
   
   Target:
   - Trucking accountants (referral program: 20% commission)
   - Freight brokers (recommend to owner-operators)
   - QuickBooks ProAdvisors (integration partners)
   - Trucking associations (sponsor newsletters)
```

### Paid Ads Strategy (Optional):

```
GOOGLE ADS:

Campaign 1: Search Ads
Keywords:
- "trucking receipt management software"
- "fuel receipt tracker"
- "QuickBooks receipt scanner for trucking"
- "expense tracking for owner operators"

Budget: $20/day ($600/month)
Target CPC: $3-$5
Expected clicks: 120-200/month
Expected signups: 12-20 (10% conversion)

Campaign 2: Display Ads
Target: Websites about trucking, logistics, small business
Budget: $10/day ($300/month)

FACEBOOK/LINKEDIN ADS:

Target Audience:
- Job titles: Owner, CEO, Office Manager
- Industries: Trucking, Logistics, Transportation
- Company size: 1-50 employees
- Interests: QuickBooks, trucking

Ad Creative:
- Image: Dashboard screenshot
- Headline: "Stop Wasting 20 Hours on Receipt Entry"
- CTA: "Start Free Trial"

Budget: $15/day ($450/month)
Expected signups: 5-10/month

TOTAL PAID AD SPEND: $1,350/month
Expected customers: 15-30 signups → 7-15 paid (50% conversion)
CAC: $90-$193 (within acceptable range if LTV = $1,896)
```

### Validation Checklist:
- [ ] Sent 50+ cold emails
- [ ] Posted 3+ blog articles
- [ ] Created 2+ YouTube videos
- [ ] Active in LinkedIn groups
- [ ] Engaged on Reddit
- [ ] Set up referral program
- [ ] Launched paid ads (optional)
- [ ] Tracked acquisition sources (Mixpanel)

---

## Summary: Phase 5 Complete ✅

**What You've Built:**
- ✅ Professional landing page
- ✅ SEO optimization (meta tags, sitemap, structured data)
- ✅ Analytics tracking (Vercel, Mixpanel)
- ✅ Security hardening (auth, rate limiting, encryption)
- ✅ Production deployment checklist
- ✅ Customer acquisition plan (cold email, content, ads)

**YOU'RE READY TO LAUNCH! 🚀**

---

## 🎊 LAUNCH DAY CHECKLIST

### T-24 Hours:
- [ ] Final security audit
- [ ] Database backup
- [ ] Test payment flow (Stripe live mode)
- [ ] Verify all emails sending
- [ ] Check uptime monitoring

### Launch Morning:
- [ ] Deploy to production
- [ ] Verify DNS propagation
- [ ] Test signup flow (create test account)
- [ ] Upload test receipt (verify OCR)
- [ ] Check Stripe webhooks working
- [ ] Monitor error logs (Sentry)

### Launch Announcement:
- [ ] Post on LinkedIn: "Excited to launch FiscalNinja!"
- [ ] Email beta users: "We're live!"
- [ ] Post in trucking Facebook groups
- [ ] Submit to Product Hunt (optional)
- [ ] Tweet launch announcement

### First 48 Hours:
- [ ] Monitor signups in real-time
- [ ] Respond to support emails <2 hours
- [ ] Fix critical bugs immediately
- [ ] Track conversion funnel
- [ ] Collect user feedback

### Week 1 Goals:
- [ ] 10 free trial signups
- [ ] 3 paying customers
- [ ] 50 receipts processed
- [ ] <5% churn
- [ ] No critical errors

---

## 🎯 POST-LAUNCH: Path to $10k MRR

### Month 1-3: Validate Product-Market Fit
- **Goal:** 10 paying customers, $500 MRR
- **Focus:** Fix bugs, improve OCR accuracy, gather feedback
- **Channels:** Manual outreach (LinkedIn, cold email)

### Month 4-6: Scale Acquisition
- **Goal:** 30 customers, $2,000 MRR
- **Focus:** Content marketing (blog posts, videos)
- **Channels:** SEO, YouTube, partnerships

### Month 7-12: Growth & Optimization
- **Goal:** 70-100 customers, $5,000-$7,000 MRR
- **Focus:** Paid ads, referral program, features (mobile app)
- **Channels:** Google Ads, LinkedIn Ads, word-of-mouth

### Month 13-24: Hit $10k MRR
- **Goal:** 140 customers, $10,000 MRR
- **Focus:** Upsells (Solo → Fleet), expand to construction
- **Channels:** All channels + sales team (hire VA)

---

## 🛠️ NICE-TO-HAVE FEATURES (Add After Launch)

**Month 2-3:**
- [ ] WhatsApp bot (upload via text)
- [ ] Mileage tracker integration
- [ ] Driver reimbursement workflow

**Month 4-6:**
- [ ] Mobile app (React Native)
- [ ] Xero integration (alternative to QuickBooks)
- [ ] Automated tax forms (Schedule C export)

**Month 7-12:**
- [ ] API for third-party integrations
- [ ] White-label for accountants
- [ ] Multi-currency support (Canada)

---

## 📚 RESOURCES & NEXT STEPS

**Development Tools:**
- Next.js Docs: https://nextjs.org/docs
- Supabase Docs: https://supabase.com/docs
- Stripe Docs: https://stripe.com/docs
- Google Vision API: https://cloud.google.com/vision/docs

**Learning:**
- "The Mom Test" by Rob Fitzpatrick (customer interviews)
- "Traction" by Gabriel Weinberg (acquisition channels)
- "The SaaS Playbook" by Rob Walling (bootstrapping)

**Communities:**
- Indie Hackers (community for bootstrappers)
- r/SaaS (Reddit community)
- MicroConf (conference for SaaS founders)

**Tools:**
- Figma (design mockups)
- Loom (demo videos)
- Mailchimp (email marketing)
- Zapier (automation workflows)

---

## 🎓 FINAL ADVICE

1. **Ship Fast, Iterate:** Don't wait for perfection. Launch with MVP, improve based on feedback.

2. **Talk to Customers:** Spend 50% of your time talking to trucking companies, not coding.

3. **Focus on One Channel:** Master cold email OR content OR ads. Don't do everything at once.

4. **Measure Everything:** Track every signup source, conversion rate, churn. Data beats guesses.

5. **Stay Lean:** Don't hire until you hit $5k MRR. Automate as much as possible.

6. **Solve Real Pain:** If users don't say "I need this NOW," you're solving the wrong problem.

7. **Persistence Wins:** It takes 12-18 months to hit $10k MRR. Don't give up at month 3.

---

## ✅ YOU'RE READY TO BUILD!

**Next Steps:**
1. Copy LLM prompts from this notebook
2. Paste into ChatGPT/Claude to generate code
3. Follow the 30-day timeline
4. Launch on Day 30
5. Get your first paying customer
6. Scale to $10k MRR

**Good luck! 🚀**

Questions? Stuck on a step? Review the relevant phase and re-read the LLM prompts. They're designed to be copy-paste ready.

Now go build FiscalNinja!